In [1]:
documents = [
    "The economy of France is based on a highly developed market-oriented economy.",
    "Inflation in the United States has been a key issue for policymakers in 2023.",
    "The stock market showed a strong recovery after a weak quarter.",
    "Cryptocurrency regulations are being discussed in major financial hubs."
]

In [2]:
import faiss
print(faiss.__version__)
print(faiss.__file__)

1.9.0
g:\Program\anaconda3\envs\rag\Lib\site-packages\faiss\__init__.py


In [3]:
import faiss
from transformers import BertTokenizer, BertModel
import numpy as np

# 문서 임베딩 생성 함수
def get_embeddings(texts, model, tokenizer):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True)
    outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.detach().numpy()

# BERT 모델과 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# 문서 임베딩 생성
doc_embeddings = get_embeddings(documents, model, tokenizer)

# FAISS 인덱스 생성
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

# 검색 함수
def search(query, top_k=1):
    query_embedding = get_embeddings([query], model, tokenizer)
    D, I = index.search(query_embedding, top_k)
    return [documents[i] for i in I[0]]

g:\Program\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
"""
"당신은 경제 뉴스를 전달하고 분석하는 챗봇입니다.\n"
"- 당신의 역할은 사용자의 질문에 reference를 바탕으로 답변하는 것입니다.\n"
"- 만약 사용자의 질문이 reference와 관련이 없다면, {제가 가지고 있는 정보로는 답변할 수 없습니다.}라고만 반드시 말하세요.\n\n"
"""

'\n"당신은 경제 뉴스를 전달하고 분석하는 챗봇입니다.\n"\n"- 당신의 역할은 사용자의 질문에 reference를 바탕으로 답변하는 것입니다.\n"\n"- 만약 사용자의 질문이 reference와 관련이 없다면, {제가 가지고 있는 정보로는 답변할 수 없습니다.}라고만 반드시 말하세요.\n\n"\n'

In [5]:
"""
You are a chatbot that delivers and analyzes economic news.

Your role is to answer users' questions based on the given reference.
If the user's question is not related to the reference, you must only say, 'I cannot answer with the information I have.'
"""

"\nYou are a chatbot that delivers and analyzes economic news.\n\nYour role is to answer users' questions based on the given reference.\nIf the user's question is not related to the reference, you must only say, 'I cannot answer with the information I have.'\n"

In [6]:
# cell 3
from utils.deepseek_client import DeepSeekClient  # deepseek_client.py에서 DeepSeekClient import
import torch

# DeepSeekClient 인스턴스 생성
deepseek_client = DeepSeekClient(model="deepseek-r1:14b")

# 사용자 질문
query = "What is the inflation rate in 2023?"

# 관련 문서 검색
retrieved_docs = search(query, top_k=1)
context = retrieved_docs[0]

# DeepSeekClient를 사용한 답변 생성 함수
def generate_answer(query, context):
    # 역할 및 지침 추가
    role_prompt = (
        "You are a chatbot that delivers and analyzes economic news.\n"
        "Your role is to answer users' questions based on the given reference.\n"
        "If the user's question is not related to the reference, you must only say, 'I cannot answer with the information I have.'\n\n"
    )
    
    input_text = f"{role_prompt}Reference: {context}\n\nQuestion: {query}\n\nAnswer:"
    
    return deepseek_client.ask(input_text)  # DeepSeek 모델에 질의하여 응답 반환

# 답변 생성
answer = generate_answer(query, context)
print(f"Question: {query}")
print(f"Answer: {answer}")


Question: What is the inflation rate in 2023?
Answer: Error: Request timed out. The model took too long to respond.


In [38]:
# 사용자 질문
query = "What is the inflation rate in 2023?"

# 관련 문서 검색
retrieved_docs = search(query, top_k=1)
context = retrieved_docs[0]

# 답변 생성
answer = generate_answer(query, context)
print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the inflation rate in 2023?
Answer: The inflation rate is the percentage of the economy's total output that is actually growing.

Question:
